In [ ]:
import pandas as pd
import re
import spacy
import math
from collections import Counter, defaultdict

nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])

def preprocess(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r"[^\w\s]", " ", text.lower())
    return " ".join(token.lemma_ for token in nlp(text) if not token.is_space)

def build_inverted_index(corpus):
    index = defaultdict(list)
    for doc_id, text in enumerate(corpus):
        for word in set(text.split()):
            index[word].append(doc_id)
    return index

def tf_idf(query, index, corpus_counts, M):
    query_words = query.split()
    query_counts = Counter(query_words)
    scores = defaultdict(float)
    
    for word in set(query_words):
        if word in index:
            df_w = len(index[word])
            idf = math.log((M + 1) / df_w)
            for doc_id in index[word]:
                c_w_q = query_counts[word]
                c_w_d = corpus_counts[doc_id][word]
                scores[doc_id] += c_w_q * c_w_d * idf
    return scores

df = pd.read_csv("articles.csv")
df['prep_title'] = df['title'].apply(preprocess)
df['prep_text'] = df['text'].apply(preprocess)

title_counts = [Counter(text.split()) for text in df['prep_title']]
text_counts = [Counter(text.split()) for text in df['prep_text']]

title_index = build_inverted_index(df['prep_title'])
text_index = build_inverted_index(df['prep_text'])
M = len(df)

query = "coursera vs udacity machine learning"
alpha = 0.7
prep_query = preprocess(query)

title_scores = tf_idf(prep_query, title_index, title_counts, M)
text_scores = tf_idf(prep_query, text_index, text_counts, M)

final_scores = []
for doc_id in range(M):
    score = alpha * title_scores.get(doc_id, 0.0) + (1 - alpha) * text_scores.get(doc_id, 0.0)
    if score > 0:
        final_scores.append({
            'title': df.iloc[doc_id]['title'],
            'text': df.iloc[doc_id]['text'][:50] + "...",
            'scores': score
        })

results_df = pd.DataFrame(final_scores).sort_values(by='scores', ascending=False).reset_index(drop=True)
results_df.head(10)

,title,text,scores
0,Coursera vs Udacity for Machine Learning – Hac...,2018 is an exciting time for students of machi...,40.478177
1,Every single Machine Learning course on the in...,"A year and a half ago, I dropped out of one of...",31.463670
2,Every single Machine Learning course on the in...,"A year and a half ago, I dropped out of one of...",31.463670
3,Every single Machine Learning course on the in...,"A year and a half ago, I dropped out of one of...",31.463670
4,Every single Machine Learning course on the in...,"A year and a half ago, I dropped out of one of...",31.463670
5,In defense of skepticism about deep learning –...,In a recent appraisal of deep learning (Marcus...,9.878529
6,In defense of skepticism about deep learning –...,In a recent appraisal of deep learning (Marcus...,9.878529
7,Machine Learning in a Week – Learning New Stuf...,Getting into machine learning (ml) can seem li...,8.404892
8,Machine Learning in a Year – Learning New Stuf...,This is a follow up to an article I wrote last...,7.009145
9,Machine Learning is Fun! – Adam Geitgey – Medium,Update: This article is part of a series. Chec...,6.956517


In [7]:
import pickle
from datasets import load_dataset

# 1. Ultimátní falešná třída, která pohltí úplně cokoliv
class DummyDataset(dict):
    def __init__(self, *args, **kwargs):
        # Schválně nepředáváme args ani kwargs dál, vytvoříme jen prázdný slovník
        super().__init__()
        
    def __setstate__(self, state):
        pass
        
    def __setattr__(self, key, value):
        pass

# 2. Vlastní Unpickler
class SafeUnpickler(pickle.Unpickler):
    def find_class(self, module, name):
        if "datasets" in module or "pyarrow" in module:
            return DummyDataset
        return super().find_class(module, name)

# 3. Načtení dat pomocí našeho zachránce
print("Zachraňuji data z pickle souboru...")
with open("wiki_indices.pkl", "rb") as f:
    wiki_data = SafeUnpickler(f).load()

wiki_title_index = wiki_data["title_index"]
wiki_text_index = wiki_data["text_index"]
wiki_M = wiki_data["doc_count"]

# 4. Načteme si čisté titulky lokálně
print("Stahuji čisté titulky z Hugging Face...")
local_dataset = load_dataset("wikimedia/wikipedia", "20231101.simple", split="train")
wiki_titles_raw = local_dataset["title"]

print("Hotovo! Záchrana se povedla, 40 minut ušetřeno.")

Zachraňuji data z pickle souboru...
Stahuji čisté titulky z Hugging Face...


c:\Users\Ruda\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Ruda\.cache\huggingface\hub\datasets--wikimedia--wikipedia. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\Ruda\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\huggingface_hub\file_d

Hotovo! Záchrana se povedla, 40 minut ušetřeno.


In [ ]:
import math
from collections import defaultdict, Counter

def search_wikipedia(query, alpha=0.7, top_k=5):
    query_tokens = preprocess(query).split()
    query_counts = Counter(query_tokens)
    
    title_scores = defaultdict(float)
    text_scores = defaultdict(float)

    for word in set(query_tokens):
        if word in wiki_title_index:
            df_w = len(wiki_title_index[word])
            idf = math.log((wiki_M + 1) / df_w)
            for doc_id, c_w_d in wiki_title_index[word].items():
                title_scores[doc_id] += query_counts[word] * c_w_d * idf
                
    for word in set(query_tokens):
        if word in wiki_text_index:
            df_w = len(wiki_text_index[word])
            idf = math.log((wiki_M + 1) / df_w)
            for doc_id, c_w_d in wiki_text_index[word].items():
                text_scores[doc_id] += query_counts[word] * c_w_d * idf
                
    final_scores = {}
    for doc_id in set(title_scores.keys()).union(text_scores.keys()):
        final_scores[doc_id] = alpha * title_scores.get(doc_id, 0.0) + (1 - alpha) * text_scores.get(doc_id, 0.0)
        
    top_results = sorted(final_scores.items(), key=lambda x: x[1], reverse=True)[:top_k]
    print(f"Dotaz: '{query}'")
    for doc_id, score in top_results:
        print(f"[{score:.4f}] {wiki_titles_raw[doc_id]}")
    print("-" * 50)

In [11]:
search_wikipedia("machine learning neural networks")
search_wikipedia("best beer brands")
search_wikipedia("biggest cities in the world")

Dotaz: 'machine learning neural networks'
[154.1857] Sewing machine
[121.6411] Network topology
[75.8562] List of Toronto subway and RT stations
[72.7096] Artificial neural network
[68.6861] The Turk
--------------------------------------------------
Dotaz: 'best beer brands'
[91.0606] Feel Good Film Festival
[72.0946] Beer
[60.5073] Grammy Award
[58.8496] List of Shahrukh Khan's awards and nominations
[55.5341] Coen brothers
--------------------------------------------------
Dotaz: 'biggest cities in the world'
[137.9837] New York City
[126.3667] Curitiba
[99.5954] List of people from New Jersey
[94.3294] List of people from Texas
[85.9543] Human history
--------------------------------------------------
